In [1]:
%run ../../Latex_macros.ipynb

<IPython.core.display.Latex object>

# Putting it together: the landscape of reinforcement learning methods

## Where this session fits

- **Series goal**
    - choose good actions by trial and error
    - from small tables to robots and language models
- **Position: the closing session, no new method**
    - Part A built the value-based path (Bellman and dynamic programming, sampling, control, deep Q-learning)
    - Part B built the policy-gradient methods (the theorem, the weight, how far, the baseline source)
    - Part C applied them to language models and met preference data
- **The question:** how do all these methods fit together, and how do we choose among them for a new problem?
- **By the end you can**
    - place any method of the series by three questions: is a model needed, what does it learn, which policy generates its data
    - read the threads of the series (bias and variance, backup width and depth, on- and off-policy, evaluate versus improve, exploration) across both parts
    - say which equation carries each idea of the series
    - choose a method family for a new problem, and say what the series left out


# Three questions place every method

**Why this section?** There are many methods. Three questions give each one an address.

- **Is a model needed?**
    - *model-based:* a model of the transitions and rewards is used for planning (known here: value and policy iteration)
    - *model-free:* only sampled experience (everything from Monte Carlo on)
- **What does it learn?**
    - *values:* $\statevalfun$ or $\actvalfun$; the policy is read off (the greedy action)
    - *a policy:* $\pi_\theta$ directly, by gradient ascent (Part B)
    - *both:* actor-critic methods (actor-critic, PPO) learn a policy and a value (the critic)
- **Which policy generates the data?**
    - *on-policy:* the policy being learned (SARSA, REINFORCE); PPO and GRPO are *near* on-policy (a slightly older copy, corrected by the ratio)
    - *off-policy:* another policy, or fixed data (Q-learning, DQN; DPO uses a fixed preference dataset)


**The grid**

<img src="images/methods_grid.png" width=95%>

- the rows combine two questions: with a known model there is no sampled data, so the model-based row has no on-/off-policy split
- the value-based path (green and blue) sits in the left column; Part B (orange) in the middle and right; DPO (purple, Part C) is in the policy column but is not a policy-gradient method
- two kinds of empty cell: *none here* (no model-based policy method in this series) and *not covered* (off-policy actor-critic methods exist, but the series skips them)
- the three questions are listed in the order values, both, policy in the columns


## Table 1, extended: all the methods

**Why this section?** Table 1 (the value-based methods, end of Part A) condensed, then the Part B and Part C rows. Terms from earlier sessions: *p*-step TD (look $p$ steps ahead), GAE (generalized advantage estimation, a blend of depths), critic (the learned value used by an actor), ratio (new over old probability of an action).

| Method | Model? | Learns | Data from | Depth | Bias of the weight or target | Variance |
|:--|:--|:--|:--|:--|:--|:--|
| dynamic programming | yes | values | none (planning) | 1 step, all successors | uses the current estimate | none from sampling |
| Monte Carlo | no | values | policy evaluated | to the end | none | high |
| TD / Q-learning / SARSA / DQN | no | values | on-policy (SARSA) or off-policy | 1 step ($p$-step between) | while the estimate is wrong (Q-learning and DQN add *maximization bias*, from the max) | lower |
| REINFORCE | no | policy | on-policy | to the end | none | high (lower with a baseline, still high) |
| actor-critic | no | policy and value | on-policy | 1 step ($p$-step or GAE between) | from the critic | lower |
| PPO | no | policy and value | near on-policy (ratio, clip) | GAE | from the critic (clipping limits the step: slot 2, not the weight) | lower |
| GRPO | no | policy | near on-policy (group per prompt) | whole response | small, from the group mean and the scaling by the standard deviation | noisier baseline, no critic |
| RLHF | no | policy (reward from a reward model) | on-policy samples | as PPO | the reward model is a proxy | as PPO |
| DPO | no | policy (an implicit reward, no explicit reward model) | offline preference data | whole response | not comparable: a different loss, no sampled returns | shifts away from the fixed data are the risk |

- the **depth** and **bias** columns carry over from Part A to the weight of the policy gradient: the same dial


## The slot table of Part B, in the grid

**Why this section?** Part B's methods are one theorem; the rows differ only in three choices.

| Method | Slot 1: weight | Slot 2: how far | Slot 3: baseline source |
|:--|:--|:--|:--|
| raw theorem | return | plain step | none |
| REINFORCE | return minus baseline | plain step | moving average |
| actor-critic | TD error or GAE | plain step | learned critic |
| PPO | GAE with a critic | clipped ratio, several epochs | learned critic |
| GRPO | group-relative advantage | clipped ratio, several epochs | group mean |

- in the grid, all these rows are in the middle and right columns, near on-policy
- RLHF-style recipes that use PPO or GRPO make choices from this table, plus a reward source (program, reward model, judge) and usually a KL penalty to a reference model
- DPO is outside the table: it is not a variant of the policy gradient theorem (Part C)


## The threads, one last time

**Why this section?** Each thread appeared early and came back. This is the whole path of each.

- **bias and variance:** the targets of Monte Carlo and TD $\to$ the weight in the policy gradient $\to$ baseline, critic, group mean (clipping, separately, limits the step)
- **backup width and depth:** dynamic programming, Monte Carlo, $p$-step TD $\to$ return-to-go, TD error, GAE
- **on-policy versus off-policy:** light at Monte Carlo, defined at SARSA versus Q-learning $\to$ replay allowed in DQN $\to$ the ratio in PPO $\to$ offline DPO
- **evaluate versus improve:** policy iteration $\to$ control with values $\to$ the critic and the actor
- **exploration:** greedy versus $\epsilon$-greedy $\to$ a stochastic policy that explores by sampling $\to$ a group that must differ
- **the proxy problem** (new in Part C): the reward is only a stand-in, so it can be exploited (reward hacking); the KL penalty to a reference model limits how far


# Equations behind the ideas (1): value-based

**Why this section?** Each idea of Part A condensed to its equation.

| Idea | The equation |
|:--|:--|
| the return | $\rlret_\tt = \rewseq_{\tt+1} + \disc \, \rlret_{\tt+1}$ |
| the Bellman equation | $\statevalfun_\pi(\state) = \sum_\act \pi(\act \mid \state) \sum_{\state', \rew} \transp(\state', \rew \mid \state, \act) \bigl( \rew + \disc \, \statevalfun_\pi(\state') \bigr)$ |
| TD learning | $\statevalfun(\stateseq_\tt) \leftarrow \statevalfun(\stateseq_\tt) + \alpha \bigl[ \rewseq_{\tt+1} + \disc \, \statevalfun(\stateseq_{\tt+1}) - \statevalfun(\stateseq_\tt) \bigr]$ |
| SARSA | $\actvalfun(\stateseq_\tt, \actseq_\tt) \leftarrow \actvalfun(\stateseq_\tt, \actseq_\tt) + \alpha \bigl[ \rewseq_{\tt+1} + \disc \, \actvalfun(\stateseq_{\tt+1}, \actseq_{\tt+1}) - \actvalfun(\stateseq_\tt, \actseq_\tt) \bigr]$ |
| Q-learning | $\actvalfun(\stateseq_\tt, \actseq_\tt) \leftarrow \actvalfun(\stateseq_\tt, \actseq_\tt) + \alpha \bigl[ \rewseq_{\tt+1} + \disc \max{\act'} \actvalfun(\stateseq_{\tt+1}, \act') - \actvalfun(\stateseq_\tt, \actseq_\tt) \bigr]$ |


# Equations behind the ideas (2): policy-based and language models

**Why this section?** Part B and Part C in the same form. The advantage and the ratio are defined in their sessions; here they are named.

| Idea | The equation |
|:--|:--|
| the policy gradient theorem (return-to-go form) | $\nabla_\theta J(\theta) = \Exp{\tau \sim \pi_\theta} \bigl[ \sum_\tt \nabla_\theta \log \pi_\theta(\actseq_\tt \mid \stateseq_\tt) \, \rlret_{\tt,\tau} \bigr]$ |
| the advantage form | the same with the weight $\advseq_\tt$ in place of $\rlret_{\tt,\tau}$: the return minus a baseline, or an estimate such as the TD error $\delta_\tt = \rewseq_{\tt+1} + \disc \, \statevalfun(\stateseq_{\tt+1}) - \statevalfun(\stateseq_\tt)$ or GAE |
| the probability ratio | $r_\tt(\theta) = \pi_\theta(\actseq_\tt \mid \stateseq_\tt) \, / \, \pi_{\theta_{\text{old}}}(\actseq_\tt \mid \stateseq_\tt)$ |
| PPO (clipped objective) | $\min{} \bigl( r_\tt \, \hat{\advseq}_\tt, \; \text{clip}(r_\tt, \, 1 - \epsilon_{\text{clip}}, \, 1 + \epsilon_{\text{clip}}) \, \hat{\advseq}_\tt \bigr)$, with $\hat{\advseq}_\tt$ an *estimate* of the advantage ($\epsilon_{\text{clip}}$ is not the $\epsilon$ of $\epsilon$-greedy) |
| GRPO advantage | $\hat{\advseq}_i = (\rlret_i - \text{mean}) / \text{std}$ over a group of completions, in the same clipped objective |
| KL-penalized objective | maximize the objective minus $\beta \, \KL(\pi_\theta \,\|\, \pi_{\text{ref}})$, $\beta$ the weight, $\pi_{\text{ref}}$ a frozen reference model |
| DPO | $-\log \sigma\bigl(\beta [\log \tfrac{\pi_\theta(\y^+ \mid \x)}{\pi_{\text{ref}}(\y^+ \mid \x)} - \log \tfrac{\pi_\theta(\y^- \mid \x)}{\pi_{\text{ref}}(\y^- \mid \x)}]\bigr)$ on preferred and non-preferred responses |


## Choosing a family

**Why this section?** From knowing the methods to choosing among them for a new problem.

- **the model is known and small** (a board game's rules, a grid): dynamic programming
- **small discrete states and actions, model unknown:** tabular Q-learning or SARSA
- **large or continuous states, a few discrete actions:** DQN
- **continuous actions, or a stochastic policy is needed:** policy-gradient methods; PPO is a common on-policy default (off-policy methods such as SAC (soft actor-critic) are often used for continuous control and are not covered here)
- **language models:** the policy is the model; the reward comes from a program, a reward model or a judge; PPO, GRPO and preference methods are the tools
- **only fixed data, no environment:** offline methods (for preference data, DPO)
- **samples are costly or unsafe:** learn or use a model, or offline data (not covered)
- the grid's three questions decide: do we have a model? what must we learn (values or a policy)? can the data be reused (on- or off-policy)?
- in every case
    - reward design decides what the agent learns (reward hacking)
    - exploration decides what *can* be learned (a common view: RL mostly raises the probability of what the policy already tries)


## What this series left out

**Why this section?** A map should show its edges.

- **learning a model** of the environment and planning with it (model-based RL with a learned model)
- **off-policy actor-critic methods** for continuous actions (the empty cell in the grid)
- **continuous-action value methods and distributional or ensemble variants** of DQN
- **the theory** behind convergence of these methods, beyond the tabular case
- **offline RL**, **natural-gradient methods** (updates that follow the geometry of the policy distribution rather than raw parameters) and trust-region theory beyond the short TRPO mention, **maximum-entropy methods**, **importance-sampling corrections** for multi-step off-policy learning, **imitation learning**, **hierarchical RL**
- **multi-agent and partially observed problems**, beyond what the examples showed
- **the language-model details** (reward-model training, RLHF at scale, GRPO variants, the derivation and use of DPO): the Advanced course


# What we established, and what it sets up

- **Established**
    - one loop, from Frozen Lake and Microduck to language models
    - three questions place every method: model, what is learned, which policy generates the data
    - the same few threads (bias and variance, backup depth, on- and off-policy, evaluate versus improve, exploration) run through both the value-based and the policy-gradient methods, and Part C added the proxy problem
    - the equations behind the main ideas fit on two slides
- **Sets up**
    - the Advanced course: reward models, RLHF, GRPO with verifiable rewards, preference optimization
    - the code notebooks and lab exercises that implement these methods
